# 05 - Retrieval-augmented generation

Goal: load documents, chunk, embed, retrieve and generate a cited answer. Allow 45 minutes. This notebook rebuilds its own small index.

Run cells from top to bottom. Complete notebook 00 first. Each notebook starts with fresh state.


In [ ]:
from google.genai import types
from course_config import PROJECT_ID, MODEL_ID, EMBEDDING_LOCATION, make_client
client = make_client()
print('Project:', PROJECT_ID, '| Model:', MODEL_ID)


RAG provides external evidence at inference time; it does not retrain the model. Chunk overlap can preserve context but duplicates tokens. These chunks use words for clarity; production chunking should consider token counts and document structure.


In [ ]:
from lab_utils import load_documents, chunk_documents, embed_texts, retrieve, grounded_answer
embedding_client = make_client(EMBEDDING_LOCATION)
chunks = chunk_documents(load_documents(), size=35, overlap=8)
print('Chunks:', len(chunks))
print(chunks[0])
matrix = embed_texts(embedding_client, [c['text'] for c in chunks], 'RETRIEVAL_DOCUMENT')


In [ ]:
question = 'How much annual leave do I get, and how many days can I carry forward?'
hits = retrieve(embedding_client, question, chunks, matrix, k=3)
for hit in hits:
    print(hit['id'], round(hit['score'], 3), hit['text'])
answer = grounded_answer(client, question, hits)
print('\nAnswer:', answer.text)


In [ ]:
unknown = 'What is the CEO salary?'
hits = retrieve(embedding_client, unknown, chunks, matrix, k=3)
print(grounded_answer(client, unknown, hits).text)


Expected: a cited answer for leave and an abstention for salary. Inspect the actual output: grounding instructions cannot guarantee correctness. A similarity cutoff needs evaluation data; there is no universal safe threshold. Documents can contain prompt injection, so evidence must be treated as data.


## Try it yourself

Change chunk size and k. Does the answer include both leave facts? Add a fictional policy document and rebuild embeddings. Check every citation manually against its chunk.
